<img src="https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/nextia-notebook-branding/assets/courses/c36-header.png" alt="Nextia Learning · Text Classification and Routing: Send Each Message to the Right Place" width="100%">

# Few-shot examples and label descriptions

**[Few-shot examples and label descriptions](https://learning.nextia-ai.com/courses/text-routing/m05/few-shot-examples-and-label-descriptions/)** · Text Classification and Routing: Send Each Message to the Right Place · Module 5, lesson 2 of 5 · [learning.nextia-ai.com](https://learning.nextia-ai.com)

**You will learn to** give an LLM the route descriptions and a few similar labelled messages, and to measure what
changes: the accuracy, the mistakes it fixes and the mistakes it makes, and whether its written confidence is true.
The LLM answers were recorded once, so you need no key.

| | |
|---|---|
| **Time** | About 35 minutes |
| **Needs** | An internet connection for the setup cells. No account and no key: the notebook replays recorded answers. |
| **You should know** | The LLM with the route names only, from [Zero-shot classification with an LLM](https://learning.nextia-ai.com/courses/text-routing/m05/zero-shot-classification-with-an-llm/). |
| **Tested** | Python 3.12 with pandas 3.1.1, scikit-learn 1.9.1 and matplotlib 3.11.2, in Jupyter on a laptop CPU (Apple M4, 16 GB), from a fresh start (*Restart and run all*) |

### How to use this notebook

This notebook has every query, task and check of the lesson. The [lesson page](https://learning.nextia-ai.com/courses/text-routing/m05/few-shot-examples-and-label-descriptions/) has the full explanations, the diagrams and the knowledge checks that count toward your certificate. Keep both open.

1. Run the cells in order, from the top. Press **Shift+Enter** to run a cell and move to the next one.
2. At a **Predict** note, write your prediction down before you run the cell.
3. At a **Your turn** note, write your answer in the cell, run it, then run the check cell below it.
4. If you are stuck, open the [solution notebook](https://github.com/Nextia-Intelligence/nextia-learning-labs/blob/main/C36/M05-L02-few-shot-examples-and-label-descriptions/few-shot-examples-and-label-descriptions-solution.ipynb).

## Setup: install the packages

Run the next cell. It installs the versions that the course uses: pandas 3.1.1, scikit-learn 1.9.1, and matplotlib for the charts. It needs an internet connection and takes about a minute the first time.

You should see one line per package, with the versions above. If you see other versions, restart the kernel (in Colab: **Runtime › Restart session**; in Kaggle: **Run › Restart & clear cell outputs**), then run this cell again. Restarting is needed when an older version was already loaded.

In [ ]:
%pip install -q --disable-pip-version-check pandas==3.1.1 scikit-learn==1.9.1 matplotlib==3.11.2
import warnings; warnings.filterwarnings("ignore")
import pandas; print("pandas", pandas.__version__)
import sklearn; print("scikit-learn", sklearn.__version__)

## Setup: get the practice data

Run the next cell. It downloads the Larkfield inbox of this course (about 9 MB) into a folder `larkfield-inbox/data`, and the course's helper `routing.py`, and checks each file's checksum. It needs an internet connection and no account, and takes about 10 seconds.

You should see `Ready: 10000 messages (6000 train, 2000 validation, 2000 test), all synthetic.` If you run the cell again, it uses the files that are already there.

If the cell shows an error, read its last line:

- `Could not download`: check your internet connection, then run the cell again. In Kaggle, turn on **Internet** in the notebook settings.
- `wrong checksum`: delete the folder `larkfield-inbox` (in Colab: the **Files** panel on the left), then run the cell again.

The Larkfield inbox is synthetic: we made it for this course. It has no real people. The answers of the hosted models were recorded once, so you need no key.

In [ ]:
# Setup: download the course data into larkfield-inbox/ and check it.
import hashlib, os, shutil, subprocess, urllib.request
from pathlib import Path

LABS = "https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/C36/"
FILES = {
    "data/corrections.csv": "be3088aecd9b7ba5d07ea222b79e06a476806eba81bc80c2f33da695878e5cc0",
    "data/costs.json": "9a2d566f062c3382b8884ef07756f886179c69a3525d80e3d4c15c0c4fd4507e",
    "data/dataset-card.md": "ec1d7769d1d68a9e51a5b12660364c74352af936df79533a0d2170867171bdd1",
    "data/double_labelled.csv": "ba4a8dadc3c1ba8d48af893124bd33fb7a895f917a2add211486e3dbd353dfc9",
    "data/guidelines.md": "04df0cd72da4e6b7e6aa629ca21205e5395df7b66aae9268618ae14597e45782",
    "data/launch_labels_sample.csv": "600c689325b7c14e56f4a27e92e395eb0b2e965411b1abf9d724720c8b485029",
    "data/launch_month.csv": "7ef3cc2c5fa910023c6330c1e895744358846ce451e551b86be8c9abe1c4289f",
    "data/messages.csv": "266ca23c0551249fc114fc74253103ea8602a04be1fe31d1b7f91bc94ce6dd5e",
    "data/old_queue.csv": "e3cde5d19d3ebe2e6e0e067693c76796c0abbf89df0dc93dc310ad3fe42a5cae",
    "data/predictions/embeddings-newer.csv": "3c7ffe5b5371ca60c3b6bedc17b307142bcfd77f06c97db7ed267b703a50a8d2",
    "data/predictions/embeddings-newer.json": "dcd537f2971b9a48b5354cc9fbb48b87689b83b3a479da9acd119701c2fc19f6",
    "data/predictions/embeddings-older.csv": "a06c490ac17cbc1c3c75380f6cdf7775483de92961cdbb78b2a7914256926539",
    "data/predictions/embeddings-older.json": "db658bbd13861529122807726b6b41e1da9c70e3b55aba55cce9e868a795e8b4",
    "data/predictions/encoder-finetuned.csv": "abced2276671bfd413319573907a48c60952310e50978dba96c0dde0332b7d39",
    "data/predictions/encoder-finetuned.json": "1f7c89271a199c1d659bdc29d8ac0592558a1424218bd0519bbbde9840212b94",
    "data/predictions/gliclass-zero-shot.csv": "2c22e5036de94a376f7c0ccf22603c4d7c624135c8a4965123c9492748814008",
    "data/predictions/gliclass-zero-shot.json": "95ad2acafd48ad2c24516333dba7e78a3eabc7ca616c4053ce2f144f56cbaaec",
    "data/predictions/llm-few-shot.csv": "bcdaee3550f78ebb672465314bbcc9ad26283aaea69265a979739cf6a5f965e2",
    "data/predictions/llm-few-shot.json": "104b446894a0450a7e02fcd07c6db44f3e1a5486002b388374ce6dae4a744a12",
    "data/predictions/llm-zero-shot.csv": "5518e2b668a110b572687313bc13bd689774dff96c0a1614e1cd92e14ddf9ce7",
    "data/predictions/llm-zero-shot.json": "9a72ce600d0d858b00dd3b37247a305f9e8b3fc2fef3b8dbe60c0871b0c1b355",
    "data/predictions/local-llm-prompted.csv": "5bb0d84f8ee9b5086818d3525e1da3f8172b38b7875b10d1d3021b3ad5889f33",
    "data/predictions/local-llm-prompted.json": "ae328379d5ea294b3b59845ee143f24776787e748bc14c165b5e8e9ed4ff7158",
    "data/predictions/local-llm-scores.csv": "f85f570075ea9288fd14a6c3955d4310ba740814f3de472e8a263cd4f9682a5f",
    "data/predictions/local-llm-scores.json": "67a7e07ce065934a947f59c1c36d416eec76f8827f01500ce82ba6e6c4d9b99e",
    "data/predictions/nli-zero-shot.csv": "ecafb001370e5cf2904ce685683beafc86a2c290cd1da80438abb9912d099d3c",
    "data/predictions/nli-zero-shot.json": "54c2e724cb3c855a0fa6302d4ac0ab2df2ffaf788eec4ab198c6feb8d3508928",
    "data/prices.json": "9e98f9056d9d1e53d76a039da9b10d2b4fa54c8a5b35eb6cae32a2abdee7af38",
    "data/recordings/README.md": "d99eb50666045cc2ccf596e100103244fe1513b488fce46e26462a2666053ffc",
    "data/recordings/llm-few-shot.jsonl": "259a93f1d6dcbcb167cd45796c8e09a914e56c7741f14bd0aee53c996e8349d4",
    "data/recordings/llm-labels.jsonl": "cd6c670a23ee78cd3d8b51e5907ebae9d86448aa6236767c11d55227616fa8cc",
    "data/recordings/llm-reasoning.jsonl": "fc8ae631121f3240ce378df429af767ac52ea6e1560aa409de09ffc99b092300",
    "data/recordings/llm-zero-shot.jsonl": "957edfb8811e177c83bf23772f0da561659cf50728ffc95597f85cd7680aed86",
    "data/rules.json": "6f95d5e7b87869ec7975b198a3777cdd96a0db6a869554d922889479c87b2066",
    "routing.py": "8de6562d53415e5c3d8e74f44df9e8838cb71085011b29244187bde5a6f4d93c",
}
HERE = Path.cwd()
# The notebook works in its own folder, marked with a file, so it never
# writes into your own project if you open the notebook there.
if Path(".nextia-notebook").exists():  # the cell ran before: start from the parent folder
    os.chdir("..")
    HERE = Path.cwd()
PROJECT = Path("larkfield-inbox").resolve()
(PROJECT / "data").mkdir(parents=True, exist_ok=True)
(PROJECT / ".nextia-notebook").touch()

def sha256(path):
    return hashlib.sha256(path.read_bytes()).hexdigest()

def download(url, path):
    try:
        urllib.request.urlretrieve(url, path)
    except OSError:
        # Python without certificates (python.org on macOS): try curl.
        if subprocess.run(["curl", "-fsSL", "-o", str(path), url]).returncode:
            raise SystemExit(f"Could not download {url}. Check your internet connection.")

for name, expected in FILES.items():
    path = PROJECT / name
    path.parent.mkdir(parents=True, exist_ok=True)
    if path.exists() and sha256(path) == expected:
        continue
    local = HERE / name  # a copy next to the notebook, if there is one
    if local.exists():
        shutil.copy(local, path)
    else:
        download(LABS + name, path)
    if sha256(path) != expected:
        raise SystemExit(f"{name} has the wrong checksum. Delete the folder larkfield-inbox and run again.")

os.chdir(PROJECT)  # work in the project folder, as in the lesson
import csv
with open("data/messages.csv", encoding="utf-8") as f:
    splits = [row["split"] for row in csv.DictReader(f)]
print(f"Ready: {len(splits)} messages ({splits.count('train')} train, {splits.count('validation')} validation, "
      f"{splits.count('test')} test), all synthetic.")

## Setup: helpers

The next cell loads pandas as `pd`, numpy as `np` and the course helper as `rt`, reads the three parts of the inbox (`train`, `validation`, `test`), and defines the check helpers. They tell you if your answer is right, without showing the answer:

- `expect(what, yours, expected)` compares one of your numbers with the expected one, and says "too high" or "too low".
- `expect_hash(what, yours, expected)` compares any result with a hidden fingerprint of the expected one.

Run the cell. It prints nothing.

In [ ]:
# Helpers: the data, the course helper, and checks that do not show the answer.
import hashlib, json, warnings
import numpy as np
import pandas as pd
import routing as rt
warnings.filterwarnings("ignore")
pd.set_option("display.max_colwidth", 120)
pd.set_option("display.width", 160)

messages = pd.read_csv("data/messages.csv")
train = messages[messages["split"] == "train"].reset_index(drop=True)
validation = messages[messages["split"] == "validation"].reset_index(drop=True)
test = messages[messages["split"] == "test"].reset_index(drop=True)

def fingerprint(value):
    def norm(v):
        if hasattr(v, "to_dict") and hasattr(v, "columns"):
            return {"columns": [str(c) for c in v.columns], "rows": [norm(list(r)) for r in v.itertuples(index=False)]}
        if hasattr(v, "tolist"):
            return norm(v.tolist())
        if isinstance(v, dict):
            return {str(k): norm(x) for k, x in sorted(v.items(), key=lambda kv: str(kv[0]))}
        if isinstance(v, (list, tuple)):
            return [norm(x) for x in v]
        if isinstance(v, bool) or v is None or isinstance(v, int):
            return v
        if isinstance(v, float):
            return None if v != v else round(v, 4)
        if hasattr(v, "item"):
            return norm(v.item())
        return str(v)
    return hashlib.sha256(json.dumps(norm(value), sort_keys=True).encode()).hexdigest()[:16]

def expect(what, yours, expected, tolerance=1e-6):
    """Compare one number or value with the expected one."""
    if isinstance(yours, (np.integer, np.floating)):
        yours = yours.item()
    if yours is None or yours is Ellipsis:
        print(f"Not yet: {what} has no value. Write your answer in the cell above, then run it again.")
        return
    if isinstance(expected, float) and isinstance(yours, (int, float)):
        same = abs(yours - expected) <= tolerance * max(1, abs(expected))
    else:
        same = yours == expected
    if same:
        print(f"Check passed: {what} is {yours}.")
    elif isinstance(expected, (int, float)) and isinstance(yours, (int, float)):
        print(f"Not yet: {what} is {yours}, which is too {'high' if yours > expected else 'low'}.")
    else:
        print(f"Not yet: {what} is {yours!r}, which is not the expected value.")

def expect_hash(what, yours, expected):
    """Compare any result with the fingerprint of the expected one."""
    if yours is None or yours is Ellipsis:
        print(f"Not yet: {what} has no value. Write your answer in the cell above, then run it again.")
    elif fingerprint(yours) == expected:
        print(f"Check passed: {what} is right.")
    else:
        print(f"Not yet: {what} is not the expected result. Read the task again, and check each step.")

<img src="https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/nextia-notebook-branding/assets/section-divider-light.png" alt="" width="100%">

## 1. The prompt with descriptions

The course recorded a second answer for each validation and test message. This time, the **system prompt** had a
short description of each route, written from the labelling guidelines:

```text
You route customer messages for Larkfield, an online shop for home and garden products. Messages can be in English or French. Choose exactly one route. If a message asks for two things, choose the route of the main request (usually the first one).
Routes:
- orders: changes to an order before it ships: change or cancel an order, a missing order confirmation
- delivery: where a parcel is, late or missed deliveries, a new delivery day, a parcel marked delivered but not received
- returns_refunds: returns, exchanges, items that arrived damaged or wrong, the status of a refund after a return
- billing: charges and payments: a double or wrong charge, a failed payment, invoices, promotion codes
- product: questions about products: size, materials, stock, assembly, care and warranty, before or after buying
- account: the customer's Larkfield account: login, personal details, deleting the account, and someone else using the account
- needs_person: a person must read it first: a safety problem, a legal or press threat, or a complaint that asks for a manager
You also get labelled examples of similar messages. Answer in JSON with the route, your confidence from 0 to 1, and a reason of at most 12 words.
```

Each call also had the **8 training messages that look most like the message**, with their routes. The next cell
finds them the same way: it turns every training message into letter groups of 3 to 5 characters (TF-IDF), and
keeps the 8 with the highest similarity.

> **Predict.** The message is "Can I return the rug if I don't like the colour?". Its label is `product`: the
> guidelines send a question about the return policy **before** buying to the product team. Which routes will
> the 8 similar messages have?

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer

vec = TfidfVectorizer(analyzer="char_wb", ngram_range=(3, 5), min_df=2, sublinear_tf=True)
X_train = vec.fit_transform(train["text"])

def nearest(texts, k=8):
    """The positions in `train` of the k most similar training messages, for each text."""
    sims = (vec.transform(texts) @ X_train.T).toarray()
    return np.sort(np.argsort(-sims, axis=1, kind="stable")[:, :k], axis=1)

msg = test[test["message_id"] == "M-105011"].iloc[0]
print(msg["text"], "->", msg["route"])
train.iloc[nearest([msg["text"]])[0]][["text", "route"]]

> **Check.** All 8 similar messages ask the same question about another product, and all 8 are labelled `product`.
> The LLM with the route names only answered `returns_refunds` with a written confidence of 0.99.

## 2. The recorded answers, with and without the extra context

The next cell reads both recordings for the 2,000 test messages and scores them with the course's columns.

In [ ]:
def recorded(mode, messages):
    answers = pd.read_json(f"data/recordings/llm-{mode}.jsonl", lines=True).set_index("message_id")
    return answers.loc[messages["message_id"]].reset_index()

zero, few = recorded("zero-shot", test), recorded("few-shot", test)
rows = [rt.score("LLM, route names only (zero-shot)", test, zero["route"],
                 run_per_1000=zero["usd"].mean() * 1000, ms_per_message=zero["latency_s"].median() * 1000),
        rt.score("LLM, descriptions + 8 examples (few-shot)", test, few["route"],
                 run_per_1000=few["usd"].mean() * 1000, ms_per_message=few["latency_s"].median() * 1000)]
rt.show(rows)

> **Check.** With descriptions and examples, the accuracy is 0.97 and the total cost 116.1 dollars per
> 1,000 messages. With the route names only: 0.765 and 1049.0.

## 3. Measure the change, not the feeling

An overall number hides two movements: messages that the new prompt fixes, and messages that it breaks. Count both.

In [ ]:
zero_ok = zero["route"].to_numpy() == test["route"].to_numpy()
few_ok = few["route"].to_numpy() == test["route"].to_numpy()
pd.crosstab(pd.Series(zero_ok, name="zero-shot right"), pd.Series(few_ok, name="few-shot right"))

> **Check.** The few-shot prompt fixes 420 messages and breaks 11.

## 4. Your turn: is the LLM only copying its examples?

The 8 examples are close to the message. Perhaps the LLM only takes the most common route among them. Test that
idea without the LLM: for each test message, give the route that most of its 8 nearest training messages have
(a **majority vote**; on a tie, the first route in `rt.ROUTES` wins).

> **Your turn.** The cell below finds the 8 neighbours of every test message. Complete `vote`: one route per test
> message. Then the check cell scores it.

In [ ]:
neigh = nearest(test["text"])            # 2,000 rows of 8 positions in train
neigh_routes = train["route"].to_numpy()[neigh]
print(neigh_routes[:2])

vote = ...  # one route per test message: the most common route of its 8 neighbours

In [ ]:
if vote is ...:
    print("Not yet: complete vote in the cell above.")
else:
    vote_acc = rt.score("8 nearest training messages, majority vote", test, vote)["accuracy"]
    expect("the accuracy of the vote", vote_acc, 0.902, tolerance=0.002)
    print("Messages that the LLM gets right and the vote gets wrong:",
          int((few_ok & (np.array(vote) != test["route"].to_numpy())).sum()))

The vote gets about 9 messages in 10 right: similar messages are a strong signal. But the LLM does better, and it
gets many messages right where the vote is wrong. It reads the message, the descriptions and the examples together.

## 5. Failure case: a written confidence is a number in the text

Both prompts asked for a confidence from 0 to 1. Is a written 0.95 right 95 times in 100? Group the answers by
their written confidence and compare with the accuracy: a **reliability table**. The **expected calibration
error** (ECE) is the mean gap, weighted by the number of messages in each group.

In [ ]:
table_zero, ece_zero = rt.reliability(zero["confidence"], zero_ok)
table_few, ece_few = rt.reliability(few["confidence"], few_ok)
print("zero-shot ECE", ece_zero, "  few-shot ECE", ece_few)
table_zero.merge(table_few, on="bin", how="outer", suffixes=("_zero", "_few"))

> **Check.** The ECE is 0.1901 with the route names only and 0.0082 with descriptions and examples.

With the route names only, the model wrote 0.9 or more for most answers and was right far less often. With the
extra context, its written confidence is close to the truth **on this inbox**. You know that only because you
checked it on 2,000 labelled messages. Check it again for every new prompt, model or kind of message.

## Next

The [lesson page](https://learning.nextia-ai.com/courses/text-routing/m05/few-shot-examples-and-label-descriptions/) has the diagrams and the knowledge checks that count toward your certificate.

<img src="https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/nextia-notebook-branding/assets/section-divider-light.png" alt="" width="100%">

**Next lesson:** [Small local LLMs](https://learning.nextia-ai.com/courses/text-routing/m05/small-local-llms/) · **This lesson:** [read it on the site](https://learning.nextia-ai.com/courses/text-routing/m05/few-shot-examples-and-label-descriptions/)

<img src="https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/nextia-notebook-branding/assets/notebook-footer-light.png" alt="Nextia Learning · learning.nextia-ai.com" width="100%">

Free, practical AI courses at [learning.nextia-ai.com](https://learning.nextia-ai.com). The practice data is synthetic and has no real people.